# Linear Probe + ECE — Corrected Accuracy for All 4 Datasets

Computes TRUE task accuracy using a linear classification head, bypassing the ImageNet class mismatch.
Steps: freeze backbone -> train linear head (3 epochs) -> evaluate accuracy + ECE -> corrected CAG.

**Prereq:** run notebooks 01-03 and 11. Requires medmnist.

In [ ]:
CIFAR_ROOT   = r'E:\cifar-10-python'
STL10_ROOT   = r'E:\stl10'
INTEL_TRAIN  = r'C:\Users\Acer\Downloads\Intel Image Classification\seg_train\seg_train'
INTEL_TEST   = r'C:\Users\Acer\Downloads\Intel Image Classification\seg_test\seg_test'
OUTPUT_DIR   = r'E:\decision_consistency_novelty'
CONSISTENCY_CSVS={'CIFAR-10':r'E:\decision_consistency_cifar_outputs\tables\summary_table.csv','STL-10':r'E:\decision_consistency_stl10_outputs\tables\summary_table_stl10.csv','Intel-Scene':r'E:\decision_consistency_intel_outputs\tables\summary_table_intel.csv','OCTMNIST':r'E:\decision_consistency_octmnist_outputs\tables\summary_table_octmnist.csv'}
BATCH_SIZE=128; EPOCHS=3; LR=1e-3; SEED=42

In [ ]:
import os, csv, random, gc
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from scipy.stats import pearsonr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings; warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)

In [ ]:
preprocess=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])
train_loaders,test_loaders,num_classes_map={},{},{}
cifar_train=datasets.CIFAR10(root=CIFAR_ROOT,train=True,transform=preprocess,download=False)
cifar_test=datasets.CIFAR10(root=CIFAR_ROOT,train=False,transform=preprocess,download=False)
train_loaders['CIFAR-10']=DataLoader(cifar_train,batch_size=BATCH_SIZE,shuffle=False); test_loaders['CIFAR-10']=DataLoader(torch.utils.data.Subset(cifar_test,range(1000)),batch_size=BATCH_SIZE); num_classes_map['CIFAR-10']=10
stl_train=datasets.STL10(root=STL10_ROOT,split='train',transform=preprocess,download=False); stl_test=datasets.STL10(root=STL10_ROOT,split='test',transform=preprocess,download=False)
train_loaders['STL-10']=DataLoader(stl_train,batch_size=BATCH_SIZE,shuffle=False); test_loaders['STL-10']=DataLoader(torch.utils.data.Subset(stl_test,range(1000)),batch_size=BATCH_SIZE); num_classes_map['STL-10']=10
intel_train=datasets.ImageFolder(root=INTEL_TRAIN,transform=preprocess); intel_test=datasets.ImageFolder(root=INTEL_TEST,transform=preprocess)
all_idx=list(range(len(intel_test))); random.seed(SEED); random.shuffle(all_idx)
train_loaders['Intel-Scene']=DataLoader(intel_train,batch_size=BATCH_SIZE,shuffle=False); test_loaders['Intel-Scene']=DataLoader(torch.utils.data.Subset(intel_test,all_idx[:1000]),batch_size=BATCH_SIZE); num_classes_map['Intel-Scene']=6
try:
    from medmnist import OCTMNIST
    oct_train=OCTMNIST(split='train',transform=transforms.Compose([transforms.Grayscale(3),preprocess]),download=True)
    oct_test=OCTMNIST(split='test',transform=transforms.Compose([transforms.Grayscale(3),preprocess]),download=True)
    train_loaders['OCTMNIST']=DataLoader(oct_train,batch_size=BATCH_SIZE,shuffle=False); test_loaders['OCTMNIST']=DataLoader(torch.utils.data.Subset(oct_test,range(min(1000,len(oct_test)))),batch_size=BATCH_SIZE); num_classes_map['OCTMNIST']=4
    print('OCTMNIST loaded.')
except ImportError: print('medmnist not installed - skip OCTMNIST')
print('Datasets ready:',list(train_loaders.keys()))

In [ ]:
def get_extractor(model,mn):
    if 'ResNet' in mn: return nn.Sequential(*list(model.children())[:-1])
    elif 'VGG' in mn:
        class VGGF(nn.Module):
            def __init__(self,m): super().__init__(); self.f=m.features; self.a=m.avgpool
            def forward(self,x): return self.a(self.f(x)).view(x.size(0),-1)
        return VGGF(model)
    else:
        class MNF(nn.Module):
            def __init__(self,m): super().__init__(); self.f=m.features
            def forward(self,x): return F.adaptive_avg_pool2d(self.f(x),(1,1)).view(x.size(0),-1)
        return MNF(model)
def compute_ece(confs,corrects,n_bins=10):
    bins=np.linspace(0,1,n_bins+1); ece=0.0
    for i in range(n_bins):
        mask=(confs>=bins[i])&(confs<bins[i+1])
        if mask.sum()==0: continue
        ece+=np.abs(np.mean(confs[mask])-np.mean(corrects[mask]))*(mask.sum()/len(confs))
    return ece
model_fns_map={'ResNet-18':models.resnet18,'ResNet-50':models.resnet50,'VGG-16':models.vgg16,'MobileNetV2':models.mobilenet_v2}
probe_results=[]
for ds_name in train_loaders:
    for mn,fn in model_fns_map.items():
        backbone=fn(pretrained=True).eval().to(device)
        for p in backbone.parameters(): p.requires_grad=False
        extractor=get_extractor(backbone,mn).eval().to(device)
        dummy=next(iter(train_loaders[ds_name]))[0][:1].to(device)
        with torch.no_grad(): feat_dim=extractor(dummy).view(1,-1).shape[1]
        del dummy; head=nn.Linear(feat_dim,num_classes_map[ds_name]).to(device); optimizer=optim.Adam(head.parameters(),lr=LR); criterion=nn.CrossEntropyLoss()
        head.train()
        for epoch in range(EPOCHS):
            for imgs,labels in tqdm(train_loaders[ds_name],desc=f'{ds_name}/{mn} ep{epoch+1}',leave=False):
                imgs=imgs.to(device); labels=labels.squeeze().long().to(device)
                with torch.no_grad(): feats=extractor(imgs)
                if feats.ndim>2: feats=feats.view(feats.size(0),-1)
                optimizer.zero_grad(); loss=criterion(head(feats),labels); loss.backward(); optimizer.step(); del feats,loss
        head.eval(); all_confs,all_corrects=[],[]
        with torch.no_grad():
            for imgs,labels in test_loaders[ds_name]:
                imgs=imgs.to(device); labels=labels.squeeze().long().to(device); feats=extractor(imgs)
                if feats.ndim>2: feats=feats.view(feats.size(0),-1)
                probs=F.softmax(head(feats),dim=1); confs,preds=probs.max(dim=1)
                all_confs.append(confs.cpu().numpy()); all_corrects.append((preds==labels).cpu().numpy()); del feats
        confs_np=np.concatenate(all_confs); corr_np=np.concatenate(all_corrects)
        acc=corr_np.mean(); ece=compute_ece(confs_np,corr_np)
        probe_results.append({'Dataset':ds_name,'Model':mn,'Accuracy':round(acc,4),'ECE':round(ece,4)})
        print(f'{ds_name:14} {mn:14} Acc={acc:.3f}  ECE={ece:.3f}')
        del backbone,extractor,head,optimizer; gc.collect(); torch.cuda.empty_cache()
csv_path=os.path.join(OUTPUT_DIR,'accuracy_ece_linear_probe.csv')
with open(csv_path,'w',newline='') as f: w=csv.DictWriter(f,fieldnames=['Dataset','Model','Accuracy','ECE']); w.writeheader(); w.writerows(probe_results)
print('Saved:',csv_path)

In [ ]:
def load_cs(path):
    if not os.path.exists(path): return {}
    with open(path) as f: return {(r['Dataset'],r['Model']):float(r['Avg_Consistency_Score']) for r in csv.DictReader(f)}
cs_map={}; [cs_map.update(load_cs(path)) for path in CONSISTENCY_CSVS.values()]
final_rows=[]
for row in probe_results:
    cs=cs_map.get((row['Dataset'],row['Model']))
    if cs is None: continue
    final_rows.append({**row,'Avg_CS':round(cs,4),'CAG':round(row['Accuracy']-cs,4)})
print(f'{"Dataset":<14} {"Model":<14} {"Acc":>7} {"ECE":>7} {"CS":>7} {"CAG":>8}'); print('-'*65)
for r in sorted(final_rows,key=lambda x:x['Dataset']): print(f"{r['Dataset']:<14} {r['Model']:<14} {r['Accuracy']:7.3f} {r['ECE']:7.3f} {r['Avg_CS']:7.3f} {r['CAG']:8.4f}")
final_csv=os.path.join(OUTPUT_DIR,'final_accuracy_ece_cag_4datasets.csv')
with open(final_csv,'w',newline='') as f: w=csv.DictWriter(f,fieldnames=['Dataset','Model','Accuracy','ECE','Avg_CS','CAG']); w.writeheader(); w.writerows(final_rows)
print('Saved:',final_csv)

In [ ]:
if final_rows:
    ece_v=[r['ECE'] for r in final_rows]; cs_v=[r['Avg_CS'] for r in final_rows]; r_val,p_val=pearsonr(ece_v,cs_v)
    fig,axes=plt.subplots(1,2,figsize=(12,5))
    axes[0].scatter(ece_v,cs_v,color='steelblue',s=70); [axes[0].annotate(f"{row['Dataset'][:4]} {row['Model'][:4]}",(row['ECE'],row['Avg_CS']),fontsize=7) for row in final_rows]
    axes[0].set_xlabel('ECE'); axes[0].set_ylabel('CS'); axes[0].set_title(f'ECE vs CS  r={r_val:.2f} p={p_val:.3f}'); axes[0].grid(alpha=0.3)
    sorted_r=sorted(final_rows,key=lambda x:x['CAG']); labels=[f"{r['Dataset'][:4]} {r['Model'][:6]}" for r in sorted_r]; cag_v=[r['CAG'] for r in sorted_r]
    axes[1].bar(labels,cag_v,color=['tomato' if v>0 else 'steelblue' for v in cag_v]); axes[1].axhline(0,color='black',lw=1.5); axes[1].set_ylabel('CAG'); axes[1].tick_params(axis='x',rotation=45); axes[1].set_title('CAG Dichotomy — All Configurations'); axes[1].grid(axis='y',alpha=0.3)
    plt.tight_layout(); path=os.path.join(OUTPUT_DIR,'ece_cag_final_16configs.png'); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',path)